In [1]:
import pandas as pd

data = pd.read_csv('./Updated Data/power_data_handle.csv')
data.info()
data = data.dropna()
data.info()
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader
import torch
import torch.nn as nn
import torch.optim as optim

#处理数据
X = data[['type_no', 'length', 'breadth', 'gross_tonnage', 'deadweight']].values
y = data[['engine_power']].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
train_dataset = TensorDataset(torch.Tensor(X_train_scaled), torch.Tensor(y_train))
test_dataset = TensorDataset(torch.Tensor(X_test_scaled), torch.Tensor(y_test))

#数据loader
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10043 entries, 0 to 10042
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Unnamed: 0.1   10043 non-null  int64  
 1   Unnamed: 0     10043 non-null  int64  
 2   type_no        10043 non-null  int64  
 3   length         10043 non-null  float64
 4   breadth        10043 non-null  float64
 5   draught        10043 non-null  float64
 6   gross_tonnage  10043 non-null  float64
 7   deadweight     10043 non-null  float64
 8   engine_power   10043 non-null  float64
dtypes: float64(6), int64(3)
memory usage: 706.3 KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10043 entries, 0 to 10042
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Unnamed: 0.1   10043 non-null  int64  
 1   Unnamed: 0     10043 non-null  int64  
 2   type_no        10043 non-null  int64  
 3   length         10043 no

D:\Anaconda3\envs\ai\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 定义模型
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(5, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, 1)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        x = self.fc4(x)
        return x

#init
model = MLP()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 训练模型
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

epochs = 2000
train_losses = []
test_losses = []



In [4]:
for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    model.eval()
    test_loss = 0.0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
    
            test_loss += loss.item()
    
    train_losses.append(train_loss / len(train_loader))
    test_losses.append(test_loss / len(test_loader))
    
    if (epoch+1) % 100 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Train Loss: {train_losses[-1]:.4f}, Test Loss: {test_losses[-1]:.4f}')
        
# # 绘制损失曲线
# plt.plot(range(1, epochs+1), train_losses, label='Train Loss')
# plt.plot(range(1, epochs+1), test_losses, label='Test Loss')
# plt.xlabel('Epoch')
# plt.ylabel('Loss')
# plt.legend()
# plt.show()



Epoch [100/2000], Train Loss: 36635350.2814, Test Loss: 31678236.5675
Epoch [200/2000], Train Loss: 33933096.1137, Test Loss: 29047052.6468
Epoch [300/2000], Train Loss: 32933084.3229, Test Loss: 28140070.0942
Epoch [400/2000], Train Loss: 31634811.0338, Test Loss: 26788254.1796
Epoch [500/2000], Train Loss: 30982008.4669, Test Loss: 25866226.2927
Epoch [600/2000], Train Loss: 30535538.5149, Test Loss: 25151403.1801
Epoch [700/2000], Train Loss: 30108297.7747, Test Loss: 23761355.8145
Epoch [800/2000], Train Loss: 29479678.9806, Test Loss: 24397203.4375
Epoch [900/2000], Train Loss: 28959210.7284, Test Loss: 24760280.6260
Epoch [1000/2000], Train Loss: 28482579.1512, Test Loss: 24428855.2793
Epoch [1100/2000], Train Loss: 28631505.7757, Test Loss: 23560594.3244
Epoch [1200/2000], Train Loss: 27825509.4461, Test Loss: 24502307.9062
Epoch [1300/2000], Train Loss: 27716048.6262, Test Loss: 24164980.7192
Epoch [1400/2000], Train Loss: 27651426.0287, Test Loss: 22957805.7153
Epoch [1500/200

In [5]:
#测试集
model.eval()
with torch.no_grad():
    y_pred = model(torch.Tensor(X_test_scaled).to(device))
    y_pred_cpu = y_pred.cpu().numpy()

#计算指标
from sklearn.metrics import mean_squared_error, r2_score
mse = mean_squared_error(y_test, y_pred_cpu)
r2 = r2_score(y_test, y_pred_cpu)

print(f"MSE：{mse},R2：{r2}")


MSE：24513106.68435755,R2：0.903466802308773


# 模型2

In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import torch

data = pd.read_csv('./Updated Data/power_data_handle.csv')
X = data[['type_no', 'length', 'breadth', 'gross_tonnage', 'deadweight']].values
# X = data[[ 'length', 'breadth', 'gross_tonnage', 'deadweight']].values
y = data['engine_power'].values
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

#随机森林模型
rf_model = RandomForestRegressor(n_estimators=100)


X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

# 训练
rf_model.fit(X_train, y_train)

#测试集预测
y_pred_rf = rf_model.predict(X_test)
y_pred_rf_tensor = torch.tensor(y_pred_rf, dtype=torch.float32).view(-1, 1)

#评估指标
rf_mse = mean_squared_error(y_test, y_pred_rf)
rf_r2 = r2_score(y_test, y_pred_rf)
rf_mae= mean_absolute_error(y_test, y_pred_rf)

# 输出结果
print('MSE：', rf_mse)
print('R2：', rf_r2)
print('MAE', rf_mae)



MSE： 20858169.059249286
R2： 0.9178600336872675
MAE 1587.4798572710497


In [10]:
import numpy as np
print(max(y_test))
print(min(y_test))
print(np.mean(y_test))

97575.0
1470.0
15183.418616226978


In [74]:
# import pandas as pd
# from sklearn.model_selection import train_test_split, GridSearchCV
# from sklearn.ensemble import RandomForestRegressor
# from sklearn.metrics import mean_squared_error, r2_score
# from sklearn.preprocessing import StandardScaler

# data = pd.read_csv('./Updated Data/power_data_handle.csv')

# # 提取数据
# # X = data[['type_no', 'length', 'breadth', 'gross_tonnage', 'deadweight']].values
# X = data[[ 'length', 'breadth', 'gross_tonnage', 'deadweight']].values
# y = data['engine_power'].values

# # 划分训练集和测试集
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, random_state=42)
# scaler = StandardScaler()
# X_train = scaler.fit_transform(X_train)
# X_test = scaler.transform(X_test)

# # 定义参数
# param_grid = {
#    'n_estimators': [100, 200, 300],
#    'max_depth': [None, 10, 20],
#    'min_samples_split': [2, 5, 10],
#    'min_samples_leaf': [1, 2, 4]
# }

# rf_model = RandomForestRegressor(random_state=42)


# grid_search = GridSearchCV(estimator=rf_model, param_grid=param_grid, cv=5, scoring='neg_mean_squared_error')
# grid_search.fit(X_train, y_train)

# best_params = grid_search.best_params_
# # print("最佳参数: ", best_params)

# # 使用最佳参数重新训练模型
# best_rf_model = RandomForestRegressor(**best_params, random_state=42)
# best_rf_model.fit(X_train, y_train)

# # 使用测试集进行预测
# y_pred_rf = best_rf_model.predict(X_test)

# # 计算评估指标
# rf_mse = mean_squared_error(y_test, y_pred_rf)
# rf_r2 = r2_score(y_test, y_pred_rf)

# # 输出结果
# print('MSE：', rf_mse)
# print('R^2：', rf_r2)

KeyboardInterrupt: 

In [26]:
import pickle

# 保存模型
save_path = './rf_model.pkl'
with open(save_path, 'wb') as f:
    pickle.dump(rf_model, f)
    
# 加载模型
with open('./rf_model.pkl', 'rb') as f:
    loaded_rf_model = pickle.load(f)

# 特征值
new_data = [[135, 15350, 20, 10000, 20000]]

# 使用加载的模型进行预测
predicted_value = loaded_rf_model.predict(new_data)
print("预测值：", predicted_value)

预测值： [7839.08133333]


# 模型3

In [67]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import mean_squared_error, r2_score

data = pd.read_csv('./Updated Data/power_data_handle.csv')

# 提取特征和目标变量
X = data[['length', 'breadth', 'gross_tonnage', 'deadweight']].values
y = data['engine_power'].values
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

In [68]:
#Transformer 模型
class TransformerModel(nn.Module):
    def __init__(self, input_dim, nhead, hidden_dim, num_layers, output_dim):
        super(TransformerModel, self).__init__()
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=input_dim, nhead=nhead, dim_feedforward=hidden_dim),
            num_layers=num_layers
        )
        self.fc = nn.Linear(input_dim, output_dim)

    def forward(self, x):
        x = self.transformer(x)
        x = self.fc(x)
        return x

# 定义模型参数
input_dim = 4 
nhead = 2  # 多头注意力头数
hidden_dim = 128  # 隐藏层维度
num_layers = 3  # Transformer层数
output_dim = 1  

# 初始化模型
model = TransformerModel(input_dim, nhead, hidden_dim, num_layers, output_dim)

# 定义损失函数和优化器
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [69]:
# 训练模型
num_epochs = 50000
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

for epoch in range(num_epochs):
    running_loss = 0.0
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    epoch_loss = running_loss / len(train_loader)
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {epoch_loss:.4f}')

# 模型评估
model.eval()
test_loss = 0.0
with torch.no_grad():
    predictions = []
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        predictions.extend(outputs.cpu().numpy())
        loss = criterion(outputs, targets)
        test_loss += loss.item()

test_loss /= len(test_loader)

# 计算评估指标
mse = mean_squared_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

# 输出评估指标
print(f'测试集的平均误差平方 (MSE): {mse}')
print(f'测试集的决定系数 (R^2): {r2}')

Epoch [10/50000], Loss: 491247988.5714
Epoch [20/50000], Loss: 487978024.7619
Epoch [30/50000], Loss: 483202065.7778
Epoch [40/50000], Loss: 479756635.4286
Epoch [50/50000], Loss: 475718312.0000
Epoch [60/50000], Loss: 468709869.9683
Epoch [70/50000], Loss: 461351150.0952
Epoch [80/50000], Loss: 454660961.3968
Epoch [90/50000], Loss: 446808544.0000
Epoch [100/50000], Loss: 438429698.2857
Epoch [110/50000], Loss: 429817390.9841
Epoch [120/50000], Loss: 420361231.8730
Epoch [130/50000], Loss: 410759340.6349
Epoch [140/50000], Loss: 400851733.5238
Epoch [150/50000], Loss: 391822608.8889
Epoch [160/50000], Loss: 380597851.4286
Epoch [170/50000], Loss: 370249481.7778
Epoch [180/50000], Loss: 360459262.4127
Epoch [190/50000], Loss: 349697062.7302
Epoch [200/50000], Loss: 339042435.3016
Epoch [210/50000], Loss: 328532681.2063
Epoch [220/50000], Loss: 319483973.7778
Epoch [230/50000], Loss: 308901712.8889
Epoch [240/50000], Loss: 300497047.1111
Epoch [250/50000], Loss: 291765924.0635
Epoch [26

Epoch [2080/50000], Loss: 46907046.0238
Epoch [2090/50000], Loss: 46588680.5635
Epoch [2100/50000], Loss: 46942343.3651
Epoch [2110/50000], Loss: 46339426.5873
Epoch [2120/50000], Loss: 46143051.3413
Epoch [2130/50000], Loss: 48005304.9683
Epoch [2140/50000], Loss: 47160748.2778
Epoch [2150/50000], Loss: 46665779.6508
Epoch [2160/50000], Loss: 47856377.1667
Epoch [2170/50000], Loss: 47215123.1746
Epoch [2180/50000], Loss: 46787572.3571
Epoch [2190/50000], Loss: 47222724.7778
Epoch [2200/50000], Loss: 47123273.3333
Epoch [2210/50000], Loss: 46671710.9921
Epoch [2220/50000], Loss: 46467760.8095
Epoch [2230/50000], Loss: 46570807.4286
Epoch [2240/50000], Loss: 47356114.8889
Epoch [2250/50000], Loss: 46714203.3413
Epoch [2260/50000], Loss: 46273101.1984
Epoch [2270/50000], Loss: 47046422.6825
Epoch [2280/50000], Loss: 46637647.1508
Epoch [2290/50000], Loss: 46198220.0476
Epoch [2300/50000], Loss: 46763901.8730
Epoch [2310/50000], Loss: 46443513.7540
Epoch [2320/50000], Loss: 46529939.9365


Epoch [4130/50000], Loss: 46412578.3254
Epoch [4140/50000], Loss: 46055676.9127
Epoch [4150/50000], Loss: 46345122.6667
Epoch [4160/50000], Loss: 46326502.8810
Epoch [4170/50000], Loss: 46667980.3968
Epoch [4180/50000], Loss: 46506502.8413
Epoch [4190/50000], Loss: 46914945.2778
Epoch [4200/50000], Loss: 46265990.5675
Epoch [4210/50000], Loss: 46624497.8333
Epoch [4220/50000], Loss: 46932689.2698
Epoch [4230/50000], Loss: 46000794.5397
Epoch [4240/50000], Loss: 45446950.4365
Epoch [4250/50000], Loss: 45592242.4603
Epoch [4260/50000], Loss: 46809241.7857
Epoch [4270/50000], Loss: 45911557.1190
Epoch [4280/50000], Loss: 45659884.7381
Epoch [4290/50000], Loss: 47534635.7937
Epoch [4300/50000], Loss: 46959601.6667
Epoch [4310/50000], Loss: 45361037.1825
Epoch [4320/50000], Loss: 47350989.2222
Epoch [4330/50000], Loss: 45767188.2619
Epoch [4340/50000], Loss: 44997099.5079
Epoch [4350/50000], Loss: 48641991.3968
Epoch [4360/50000], Loss: 49461359.5000
Epoch [4370/50000], Loss: 46762637.8333


Epoch [6180/50000], Loss: 46068289.3175
Epoch [6190/50000], Loss: 46400054.9444
Epoch [6200/50000], Loss: 46399263.8889
Epoch [6210/50000], Loss: 46440166.6825
Epoch [6220/50000], Loss: 46340545.4286
Epoch [6230/50000], Loss: 47689735.7698
Epoch [6240/50000], Loss: 45393722.7698
Epoch [6250/50000], Loss: 46174037.4365
Epoch [6260/50000], Loss: 46497956.3730
Epoch [6270/50000], Loss: 47069239.8730
Epoch [6280/50000], Loss: 46061038.4921
Epoch [6290/50000], Loss: 46174835.8730
Epoch [6300/50000], Loss: 46927953.1270
Epoch [6310/50000], Loss: 46429199.7857
Epoch [6320/50000], Loss: 45864558.4841
Epoch [6330/50000], Loss: 46103847.1270
Epoch [6340/50000], Loss: 45843601.1429
Epoch [6350/50000], Loss: 46484271.6667
Epoch [6360/50000], Loss: 46935580.4444
Epoch [6370/50000], Loss: 45875973.5317
Epoch [6380/50000], Loss: 49540059.6587
Epoch [6390/50000], Loss: 46534360.9286
Epoch [6400/50000], Loss: 46217735.8730
Epoch [6410/50000], Loss: 46159468.7619
Epoch [6420/50000], Loss: 46234987.9048


Epoch [8230/50000], Loss: 45241862.4841
Epoch [8240/50000], Loss: 46642363.1905
Epoch [8250/50000], Loss: 46172915.4881
Epoch [8260/50000], Loss: 45025787.1349
Epoch [8270/50000], Loss: 46285299.1508
Epoch [8280/50000], Loss: 45090642.9762
Epoch [8290/50000], Loss: 45515127.4524
Epoch [8300/50000], Loss: 46352340.5000
Epoch [8310/50000], Loss: 45679788.0238
Epoch [8320/50000], Loss: 46103628.3016
Epoch [8330/50000], Loss: 46447333.5952
Epoch [8340/50000], Loss: 45268528.0079
Epoch [8350/50000], Loss: 45732076.2143
Epoch [8360/50000], Loss: 45441100.8095
Epoch [8370/50000], Loss: 45509703.6190
Epoch [8380/50000], Loss: 47200224.5794
Epoch [8390/50000], Loss: 46349873.1667
Epoch [8400/50000], Loss: 45632386.3254
Epoch [8410/50000], Loss: 47418510.6667
Epoch [8420/50000], Loss: 46313984.9841
Epoch [8430/50000], Loss: 47455205.8492
Epoch [8440/50000], Loss: 46572950.9524
Epoch [8450/50000], Loss: 45461325.7778
Epoch [8460/50000], Loss: 47093201.6508
Epoch [8470/50000], Loss: 46540769.1587


...

Epoch [40280/50000], Loss: 45780513.7143
Epoch [40290/50000], Loss: 46608434.2381
Epoch [40300/50000], Loss: 45051719.7937
Epoch [40310/50000], Loss: 44285904.0159
Epoch [40320/50000], Loss: 45596282.3730
Epoch [40330/50000], Loss: 46307992.5952
Epoch [40340/50000], Loss: 45581646.1270
Epoch [40350/50000], Loss: 45340762.6032
Epoch [40360/50000], Loss: 45776763.5317
Epoch [40370/50000], Loss: 44935759.3175
Epoch [40380/50000], Loss: 44716132.2401
Epoch [40390/50000], Loss: 45005732.7937
Epoch [40400/50000], Loss: 45255375.3492
Epoch [40410/50000], Loss: 45029957.5238
Epoch [40420/50000], Loss: 45729897.9206
Epoch [40430/50000], Loss: 46032600.7222
Epoch [40440/50000], Loss: 45009728.3492
Epoch [40450/50000], Loss: 45781282.1270
Epoch [40460/50000], Loss: 44881820.6429
Epoch [40470/50000], Loss: 45562791.0476
Epoch [40480/50000], Loss: 46275433.1825
Epoch [40490/50000], Loss: 45401410.9365
Epoch [40500/50000], Loss: 44331337.8730
Epoch [40510/50000], Loss: 45167822.6429
Epoch [40520/500

Epoch [42280/50000], Loss: 45667346.8889
Epoch [42290/50000], Loss: 45341816.9762
Epoch [42300/50000], Loss: 45916178.0714
Epoch [42310/50000], Loss: 44949017.7738
Epoch [42320/50000], Loss: 44956942.2698
Epoch [42330/50000], Loss: 45429021.4286
Epoch [42340/50000], Loss: 45465908.1825
Epoch [42350/50000], Loss: 45729376.8889
Epoch [42360/50000], Loss: 44648918.8254
Epoch [42370/50000], Loss: 44741669.9127
Epoch [42380/50000], Loss: 46030509.3968
Epoch [42390/50000], Loss: 45158002.1825
Epoch [42400/50000], Loss: 45662351.7381
Epoch [42410/50000], Loss: 45287530.5238
Epoch [42420/50000], Loss: 45624458.5714
Epoch [42430/50000], Loss: 46125496.6825
Epoch [42440/50000], Loss: 45824508.1032
Epoch [42450/50000], Loss: 45442389.2460
Epoch [42460/50000], Loss: 45036608.2282
Epoch [42470/50000], Loss: 45433104.5397
Epoch [42480/50000], Loss: 45289685.8889
Epoch [42490/50000], Loss: 45154534.1032
Epoch [42500/50000], Loss: 48677932.1746
Epoch [42510/50000], Loss: 45363394.9921
Epoch [42520/500

Epoch [44280/50000], Loss: 45034797.4206
Epoch [44290/50000], Loss: 46147082.6984
Epoch [44300/50000], Loss: 44630379.9683
Epoch [44310/50000], Loss: 45986048.9286
Epoch [44320/50000], Loss: 46577483.2222
Epoch [44330/50000], Loss: 45791688.5238
Epoch [44340/50000], Loss: 45684862.4960
Epoch [44350/50000], Loss: 45971041.2381
Epoch [44360/50000], Loss: 44977778.0159
Epoch [44370/50000], Loss: 45224447.4762
Epoch [44380/50000], Loss: 46598401.2381
Epoch [44390/50000], Loss: 44930749.0635
Epoch [44400/50000], Loss: 44364241.1111
Epoch [44410/50000], Loss: 45093132.7857
Epoch [44420/50000], Loss: 44422234.9802
Epoch [44430/50000], Loss: 45435705.9524
Epoch [44440/50000], Loss: 45274380.3095
Epoch [44450/50000], Loss: 45108667.5794
Epoch [44460/50000], Loss: 45093143.3333
Epoch [44470/50000], Loss: 45371811.7857
Epoch [44480/50000], Loss: 45698732.5635
Epoch [44490/50000], Loss: 44781293.5238
Epoch [44500/50000], Loss: 45108800.9921
Epoch [44510/50000], Loss: 45065108.0000
Epoch [44520/500

Epoch [46280/50000], Loss: 45380475.7183
Epoch [46290/50000], Loss: 45412952.9286
Epoch [46300/50000], Loss: 44537748.1468
Epoch [46310/50000], Loss: 45386642.1349
Epoch [46320/50000], Loss: 45357822.3968
Epoch [46330/50000], Loss: 44960049.0000
Epoch [46340/50000], Loss: 45279547.6310
Epoch [46350/50000], Loss: 45711996.0159
Epoch [46360/50000], Loss: 45725667.0714
Epoch [46370/50000], Loss: 44979077.8492
Epoch [46380/50000], Loss: 44614283.8730
Epoch [46390/50000], Loss: 44721224.8175
Epoch [46400/50000], Loss: 46713577.9841
Epoch [46410/50000], Loss: 45025973.0437
Epoch [46420/50000], Loss: 44880419.3492
Epoch [46430/50000], Loss: 44932120.6349
Epoch [46440/50000], Loss: 45659178.3492
Epoch [46450/50000], Loss: 44792406.9286
Epoch [46460/50000], Loss: 45297768.0317
Epoch [46470/50000], Loss: 44841858.1429
Epoch [46480/50000], Loss: 46268845.8175
Epoch [46490/50000], Loss: 45291458.7937
Epoch [46500/50000], Loss: 45106225.4206
Epoch [46510/50000], Loss: 44942251.9365
Epoch [46520/500

Epoch [48280/50000], Loss: 45278866.5238
Epoch [48290/50000], Loss: 45619086.8333
Epoch [48300/50000], Loss: 44126786.0159
Epoch [48310/50000], Loss: 45685575.7619
Epoch [48320/50000], Loss: 45497362.1786
Epoch [48330/50000], Loss: 44690488.0635
Epoch [48340/50000], Loss: 44316506.8532
Epoch [48350/50000], Loss: 46071512.9762
Epoch [48360/50000], Loss: 44927007.6984
Epoch [48370/50000], Loss: 45453794.8175
Epoch [48380/50000], Loss: 44970120.6746
Epoch [48390/50000], Loss: 45294224.7381
Epoch [48400/50000], Loss: 45482265.2698
Epoch [48410/50000], Loss: 48577134.8810
Epoch [48420/50000], Loss: 45259023.8571
Epoch [48430/50000], Loss: 45840681.3651
Epoch [48440/50000], Loss: 45843383.6111
Epoch [48450/50000], Loss: 45748239.4802
Epoch [48460/50000], Loss: 45051932.9048
Epoch [48470/50000], Loss: 46029110.4603
Epoch [48480/50000], Loss: 45313738.3333
Epoch [48490/50000], Loss: 45806288.0238
Epoch [48500/50000], Loss: 46015206.7698
Epoch [48510/50000], Loss: 44860835.3770
Epoch [48520/500

In [60]:
# 保存模型
model_path = './TransformerModel_engine.pth'
torch.save(model.state_dict(), model_path)

# 模型4

In [70]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

# 读取 CSV 文件 
data = pd.read_csv('./Updated Data/power_data_handle.csv')

X = data[['length', 'breadth', 'gross_tonnage', 'deadweight']].values
y = data['engine_power'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#标准化
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

class MLP(nn.Module):
    def __init__(self):
        super(DeepFeedForwardNet, self).__init__()
        self.fc1 = nn.Linear(4, 128)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(128, 64)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(64, 1)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.fc3(x)
        return x

# 初始化模型和优化器
model = MLP()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 训练模型
num_epochs = 20000
for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        model.eval()
        with torch.no_grad():
            y_pred = model(X_test)
            mse = mean_squared_error(y_test.numpy(), y_pred.numpy())
            r2 = r2_score(y_test.numpy(), y_pred.numpy())
        print(f'Epoch [{epoch + 1}/{num_epochs}], MSE: {mse:.4f}, R^2: {r2:.4f}')

#评估
model.eval()
with torch.no_grad():
    y_pred = model(X_test)
    mse = mean_squared_error(y_test.numpy(), y_pred.numpy())
    r2 = r2_score(y_test.numpy(), y_pred.numpy())

#结果
print(f'最终均方误差 (MSE): {mse:.4f}')
print(f'最终决定系数 (R^2): {r2:.4f}')

Epoch [1/20000], MSE: 484477472.0000, R^2: -0.9079
Epoch [101/20000], MSE: 482645536.0000, R^2: -0.9007
Epoch [201/20000], MSE: 468318144.0000, R^2: -0.8442
Epoch [301/20000], MSE: 427397472.0000, R^2: -0.6831
Epoch [401/20000], MSE: 356708320.0000, R^2: -0.4047
Epoch [501/20000], MSE: 271456736.0000, R^2: -0.0690
Epoch [601/20000], MSE: 199957488.0000, R^2: 0.2126
Epoch [701/20000], MSE: 160655872.0000, R^2: 0.3673
Epoch [801/20000], MSE: 145407616.0000, R^2: 0.4274
Epoch [901/20000], MSE: 138034912.0000, R^2: 0.4564
Epoch [1001/20000], MSE: 131944088.0000, R^2: 0.4804
Epoch [1101/20000], MSE: 125920232.0000, R^2: 0.5041
Epoch [1201/20000], MSE: 119726216.0000, R^2: 0.5285
Epoch [1301/20000], MSE: 113348816.0000, R^2: 0.5536
Epoch [1401/20000], MSE: 106748896.0000, R^2: 0.5796
Epoch [1501/20000], MSE: 99982712.0000, R^2: 0.6063
Epoch [1601/20000], MSE: 93081648.0000, R^2: 0.6334
Epoch [1701/20000], MSE: 86290232.0000, R^2: 0.6602
Epoch [1801/20000], MSE: 79917968.0000, R^2: 0.6853
Epo

Epoch [15701/20000], MSE: 27446998.0000, R^2: 0.8919
Epoch [15801/20000], MSE: 27433332.0000, R^2: 0.8920
Epoch [15901/20000], MSE: 27427542.0000, R^2: 0.8920
Epoch [16001/20000], MSE: 27422950.0000, R^2: 0.8920
Epoch [16101/20000], MSE: 27437828.0000, R^2: 0.8919
Epoch [16201/20000], MSE: 27444818.0000, R^2: 0.8919
Epoch [16301/20000], MSE: 27437902.0000, R^2: 0.8919
Epoch [16401/20000], MSE: 27447652.0000, R^2: 0.8919
Epoch [16501/20000], MSE: 27501606.0000, R^2: 0.8917
Epoch [16601/20000], MSE: 27558420.0000, R^2: 0.8915
Epoch [16701/20000], MSE: 27602542.0000, R^2: 0.8913
Epoch [16801/20000], MSE: 27586418.0000, R^2: 0.8914
Epoch [16901/20000], MSE: 27561214.0000, R^2: 0.8915
Epoch [17001/20000], MSE: 27550690.0000, R^2: 0.8915
Epoch [17101/20000], MSE: 27552252.0000, R^2: 0.8915
Epoch [17201/20000], MSE: 27479454.0000, R^2: 0.8918
Epoch [17301/20000], MSE: 27451826.0000, R^2: 0.8919
Epoch [17401/20000], MSE: 27448700.0000, R^2: 0.8919
Epoch [17501/20000], MSE: 27456342.0000, R^2: 

In [77]:
#保存模型
model_path = '/mlp.pth'
torch.save(model.state_dict(), model_path)